In [16]:
import glob
import numpy as np
import pandas as pd
import pandasql as ps
import polars as pl
import math
import itertools 

from xgboost import XGBClassifier
XGBOOST_AVAILABLE = True
from sklearn.model_selection import train_test_split

#matplotlib libraries
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors
import seaborn as sns

#date libraries
from dateutil import parser
from datetime import datetime, timedelta, date


#p

#pandas options
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)  

pl.Config.set_tbl_cols(-1)     # show all columns, no truncation
pl.Config.set_tbl_rows(20)     # cap rows shown (avoid dumping 7M rows)
pl.Config.set_fmt_str_lengths(50)

#matplotlib setting defaults
sns.set(font="Arial",
        rc={
 "axes.axisbelow": False,
 "axes.edgecolor": "lightgrey",
 "axes.facecolor": "None",
 "axes.grid": False,
 "axes.labelcolor": "dimgrey",
 "axes.spines.right": False,
 "axes.spines.top": False,
 "figure.facecolor": "white",
 "lines.solid_capstyle": "round",
 "patch.edgecolor": "w",
 "patch.force_edgecolor": True,
 "text.color": "dimgrey",
 "xtick.bottom": False,
 "xtick.color": "dimgrey",
 "xtick.direction": "out",
 "xtick.top": False,
 "ytick.color": "dimgrey",
 "ytick.direction": "out",
 "ytick.left": False,
 "ytick.right": False})

In [17]:
def missing_data(input_data: pl.DataFrame) -> pl.DataFrame:
    '''
    Returns a dataframe with % nulls and dtype per column.
    input: polars df
    output: polars df
    '''
    total = input_data.null_count().to_pandas().T.rename(columns={0: "Total"})
    total["Percent"] = total["Total"] / input_data.height * 100
    total["Types"] = [str(dt) for dt in input_data.dtypes]
    return total

def mape(actual, pred):
    '''
    Mean Absolute Percentage Error (MAPE)
    input: array-like actual and predicted values
    output: mape value
    '''
    actual, pred = np.array(actual), np.array(pred)
    return np.mean(np.abs((actual - pred) / actual)) * 100

In [18]:
air_data = glob.glob("bts_raw_data/*.zip")

In [19]:
import io
import zipfile

frames = []
for zip_path in air_data:
    with zipfile.ZipFile(zip_path) as archive:
        csv_name = next(name for name in archive.namelist() if name.lower().endswith('.csv'))
        with archive.open(csv_name) as csv_file:
            frames.append(pl.read_csv(io.BytesIO(csv_file.read())))

air_data = pl.concat(frames, how="diagonal_relaxed")

In [34]:
context_df = air_data.to_pandas().copy()

context_df["PredictionTime"] = pd.to_datetime(
    {
        "year": context_df["Year"],
        "month": context_df["Month"],
        "day": context_df["DayofMonth"]
    },
    errors="coerce"
)

dep_hour = context_df["CRSDepTime"] // 100
dep_minute = context_df["CRSDepTime"] % 100

context_df["PredictionTime"] = (
    context_df["PredictionTime"]
    + pd.to_timedelta(dep_hour, unit="h")
    + pd.to_timedelta(dep_minute, unit="m")
)

: 

In [20]:
# just the column names, cleanest check
print(air_data.describe())

shape: (9, 111)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ sta ┆ Yea ┆ Qua ┆ Mon ┆ Day ┆ Day ┆ Fli ┆ Rep ┆ DOT ┆ IAT ┆ Tai ┆ Fli ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ CRS ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Tax ┆ Whe ┆ Whe ┆ Tax ┆ CRS ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Can ┆ Can ┆ Div ┆ CRS ┆ Ac

In [21]:
print(list(air_data.columns))

['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'FirstDepTime', 'TotalAddGTime'

In [22]:
print(air_data.describe())

shape: (9, 111)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ sta ┆ Yea ┆ Qua ┆ Mon ┆ Day ┆ Day ┆ Fli ┆ Rep ┆ DOT ┆ IAT ┆ Tai ┆ Fli ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Ori ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ Des ┆ CRS ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Dep ┆ Tax ┆ Whe ┆ Whe ┆ Tax ┆ CRS ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Arr ┆ Can ┆ Can ┆ Div ┆ CRS ┆ Ac

In [23]:
feature_cols = [
    'Year',
    'Quarter',
    'Month',
    'DayofMonth',
    'DayOfWeek',
    'Reporting_Airline',
    'Flight_Number_Reporting_Airline',
    'Tail_Number',
    'Origin',
    'OriginState',
    'Dest',
    'DestState',
    'CRSDepTime',
    'CRSArrTime',
    'DepTimeBlk',
    'ArrTimeBlk',
    'CRSElapsedTime',
    'Distance',
    'DistanceGroup',
]

target_cols = [
    'DepDel15',
    'ArrDel15',
    'ArrDelayMinutes',
    'DepDelayMinutes',
    'Cancelled',
    'Diverted'
]

df = air_data.select(
    feature_cols + target_cols
).to_pandas()

X = df[feature_cols].copy()

categorical_cols = [
    'Reporting_Airline',
    'Tail_Number',
    'Origin',
    'OriginState',
    'Dest',
    'DestState',
    'DepTimeBlk',
    'ArrTimeBlk',
]


# Tell Pandas these columns are categorical
for col in categorical_cols:
    X[col] = X[col].astype('category')

def label(row):
    if row['Cancelled'] == 1:
        return 2

    return int(row['DepDel15'])


y = df.apply(label, axis=1)

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# AEROPROPHET CONTEXTUAL FEATURE ENGINEERING
# ============================================================

context_df = air_data.to_pandas().copy()

# ------------------------------------------------------------
# 1. Create prediction timestamp
# ------------------------------------------------------------

context_df["PredictionTime"] = pd.to_datetime(
    context_df["FlightDate"].astype(str) + " " +
    context_df["CRSDepTime"].astype(str).str.zfill(4),
    format="%Y-%m-%d %H%M",
    errors="coerce"
)

# ------------------------------------------------------------
# 2. Clean delay variables
# ------------------------------------------------------------

context_df["DepDelayMinutes"] = pd.to_numeric(
    context_df["DepDelayMinutes"],
    errors="coerce"
)

context_df["ArrDelayMinutes"] = pd.to_numeric(
    context_df["ArrDelayMinutes"],
    errors="coerce"
)

context_df["Cancelled"] = pd.to_numeric(
    context_df["Cancelled"],
    errors="coerce"
).fillna(0)

context_df["Diverted"] = pd.to_numeric(
    context_df["Diverted"],
    errors="coerce"
).fillna(0)

# ------------------------------------------------------------
# 3. Create useful binary indicators
# ------------------------------------------------------------

context_df["DepDelayed"] = (
    context_df["DepDelayMinutes"] >= 15
).astype(int)

context_df["ArrDelayed"] = (
    context_df["ArrDelayMinutes"] >= 15
).astype(int)

# ------------------------------------------------------------
# 4. Create route identifiers
# ------------------------------------------------------------

context_df["Route"] = (
    context_df["Origin"].astype(str)
    + "_"
    + context_df["Dest"].astype(str)
)

context_df["StateRoute"] = (
    context_df["OriginState"].astype(str)
    + "_"
    + context_df["DestState"].astype(str)
)

# ------------------------------------------------------------
# 5. Sort chronologically
# ------------------------------------------------------------

context_df = context_df.sort_values(
    "PredictionTime"
).reset_index(drop=True)

In [ ]:
# ============================================================
# AIRPORT ROLLING STATE
# ============================================================

airport_history = context_df[
    [
        "PredictionTime",
        "Origin",
        "DepDelayMinutes",
        "DepDelayed",
        "Cancelled",
        "Diverted"
    ]
].copy()

airport_history = airport_history.rename(
    columns={"Origin": "Airport"}
)

airport_history = airport_history.sort_values(
    "PredictionTime"
)

airport_history = airport_history.set_index(
    "PredictionTime"
)

# ------------------------------------------------------------
# Rolling windows
# ------------------------------------------------------------

windows = {
    "1h": "1h",
    "3h": "3h",
    "6h": "6h",
    "12h": "12h",
    "24h": "24h"
}

for name, window in windows.items():

    grouped = airport_history.groupby("Airport")

    airport_history[f"airport_avg_dep_delay_{name}"] = (
        grouped["DepDelayMinutes"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    airport_history[f"airport_dep_delay_rate_{name}"] = (
        grouped["DepDelayed"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    airport_history[f"airport_cancel_rate_{name}"] = (
        grouped["Cancelled"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    airport_history[f"airport_diversion_rate_{name}"] = (
        grouped["Diverted"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    airport_history[f"airport_flight_volume_{name}"] = (
        grouped["DepDelayMinutes"]
        .rolling(window)
        .count()
        .reset_index(level=0, drop=True)
    )

airport_history = airport_history.reset_index()

In [ ]:
# ============================================================
# ATTACH ORIGIN AIRPORT STATE
# ============================================================

origin_features = airport_history.copy()

origin_features = origin_features.rename(
    columns={
        "Airport": "Origin"
    }
)

feature_columns = [
    col for col in origin_features.columns
    if col.startswith("airport_")
]

origin_features = origin_features[
    ["PredictionTime", "Origin"] + feature_columns
]

context_df = context_df.merge(
    origin_features,
    on=["PredictionTime", "Origin"],
    how="left"
)

In [ ]:
# ============================================================
# REMOVE CURRENT FLIGHT FROM ITS OWN CONTEXT
# ============================================================

for col in feature_columns:
    context_df[col] = (
        context_df
        .groupby("Origin")[col]
        .shift(1)
    )

In [ ]:
# ============================================================
# DESTINATION AIRPORT ROLLING STATE
# ============================================================

destination_history = context_df[
    [
        "PredictionTime",
        "Dest",
        "ArrDelayMinutes",
        "ArrDelayed",
        "Cancelled",
        "Diverted"
    ]
].copy()

destination_history = destination_history.rename(
    columns={"Dest": "Airport"}
)

destination_history = destination_history.sort_values(
    "PredictionTime"
)

destination_history = destination_history.set_index(
    "PredictionTime"
)

for name, window in windows.items():

    grouped = destination_history.groupby("Airport")

    destination_history[f"dest_airport_avg_arr_delay_{name}"] = (
        grouped["ArrDelayMinutes"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    destination_history[f"dest_airport_arr_delay_rate_{name}"] = (
        grouped["ArrDelayed"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    destination_history[f"dest_airport_cancel_rate_{name}"] = (
        grouped["Cancelled"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    destination_history[f"dest_airport_diversion_rate_{name}"] = (
        grouped["Diverted"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

destination_history = destination_history.reset_index()

In [ ]:
destination_features = destination_history.rename(
    columns={"Airport": "Dest"}
)

destination_feature_columns = [
    col for col in destination_features.columns
    if col.startswith("dest_airport_")
]

destination_features = destination_features[
    ["PredictionTime", "Dest"] + destination_feature_columns
]

context_df = context_df.merge(
    destination_features,
    on=["PredictionTime", "Dest"],
    how="left"
)

for col in destination_feature_columns:
    context_df[col] = (
        context_df
        .groupby("Dest")[col]
        .shift(1)
    )

In [ ]:
# ============================================================
# ROUTE ROLLING STATE
# ============================================================

route_history = context_df[
    [
        "PredictionTime",
        "Route",
        "DepDelayMinutes",
        "DepDelayed",
        "Cancelled"
    ]
].copy()

route_history = route_history.sort_values(
    "PredictionTime"
)

route_history = route_history.set_index(
    "PredictionTime"
)

for name, window in windows.items():

    grouped = route_history.groupby("Route")

    route_history[f"route_avg_delay_{name}"] = (
        grouped["DepDelayMinutes"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    route_history[f"route_delay_rate_{name}"] = (
        grouped["DepDelayed"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    route_history[f"route_cancel_rate_{name}"] = (
        grouped["Cancelled"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

route_history = route_history.reset_index()

In [ ]:
route_features = route_history[
    [
        "PredictionTime",
        "Route",
        "route_avg_delay_1h",
        "route_delay_rate_1h",
        "route_cancel_rate_1h",
        "route_avg_delay_3h",
        "route_delay_rate_3h",
        "route_cancel_rate_3h",
        "route_avg_delay_6h",
        "route_delay_rate_6h",
        "route_cancel_rate_6h",
        "route_avg_delay_12h",
        "route_delay_rate_12h",
        "route_cancel_rate_12h",
        "route_avg_delay_24h",
        "route_delay_rate_24h",
        "route_cancel_rate_24h"
    ]
]

context_df = context_df.merge(
    route_features,
    on=["PredictionTime", "Route"],
    how="left"
)

route_feature_columns = [
    col for col in context_df.columns
    if col.startswith("route_")
]

for col in route_feature_columns:
    context_df[col] = (
        context_df
        .groupby("Route")[col]
        .shift(1)
    )

In [ ]:
# ============================================================
# STATE-TO-STATE ROLLING STATE
# ============================================================

state_history = context_df[
    [
        "PredictionTime",
        "StateRoute",
        "DepDelayMinutes",
        "DepDelayed",
        "Cancelled"
    ]
].copy()

state_history = state_history.sort_values(
    "PredictionTime"
)

state_history = state_history.set_index(
    "PredictionTime"
)

for name, window in windows.items():

    grouped = state_history.groupby("StateRoute")

    state_history[f"state_route_avg_delay_{name}"] = (
        grouped["DepDelayMinutes"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    state_history[f"state_route_delay_rate_{name}"] = (
        grouped["DepDelayed"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    state_history[f"state_route_cancel_rate_{name}"] = (
        grouped["Cancelled"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

state_history = state_history.reset_index()

In [ ]:
state_features = state_history[
    [
        "PredictionTime",
        "StateRoute",

        "state_route_avg_delay_1h",
        "state_route_delay_rate_1h",
        "state_route_cancel_rate_1h",

        "state_route_avg_delay_3h",
        "state_route_delay_rate_3h",
        "state_route_cancel_rate_3h",

        "state_route_avg_delay_6h",
        "state_route_delay_rate_6h",
        "state_route_cancel_rate_6h",

        "state_route_avg_delay_12h",
        "state_route_delay_rate_12h",
        "state_route_cancel_rate_12h",

        "state_route_avg_delay_24h",
        "state_route_delay_rate_24h",
        "state_route_cancel_rate_24h"
    ]
]

context_df = context_df.merge(
    state_features,
    on=["PredictionTime", "StateRoute"],
    how="left"
)

state_feature_columns = [
    col for col in context_df.columns
    if col.startswith("state_route_")
]

for col in state_feature_columns:
    context_df[col] = (
        context_df
        .groupby("StateRoute")[col]
        .shift(1)
    )

In [ ]:
df = context_df[
    feature_cols + target_cols + [
        col for col in context_df.columns
        if (
            col.startswith("airport_")
            or col.startswith("dest_airport_")
            or col.startswith("route_")
            or col.startswith("state_route_")
        )
    ]
].copy()

In [ ]:
X = df[feature_cols].copy()

In [24]:
X_engineered = X.copy()

# Scheduled departure time
X_engineered["CRSDepHour"] = (
    X_engineered["CRSDepTime"] // 100
)

X_engineered["CRSDepMinute"] = (
    X_engineered["CRSDepTime"] % 100
)

X_engineered["CRSDepMinutes"] = (
    X_engineered["CRSDepHour"] * 60
    + X_engineered["CRSDepMinute"]
)

# Scheduled arrival time
X_engineered["CRSArrHour"] = (
    X_engineered["CRSArrTime"] // 100
)

X_engineered["CRSArrMinute"] = (
    X_engineered["CRSArrTime"] % 100
)

X_engineered["CRSArrMinutes"] = (
    X_engineered["CRSArrHour"] * 60
    + X_engineered["CRSArrMinute"]
)

# Route
X_engineered["Route"] = (
    X_engineered["Origin"].astype(str)
    + "_"
    + X_engineered["Dest"].astype(str)
)

# Scheduled distance per minute
X_engineered["DistancePerMinute"] = (
    X_engineered["Distance"]
    / X_engineered["CRSElapsedTime"].replace(0, pd.NA)
)

# Weekend indicator
X_engineered["IsWeekend"] = (
    X_engineered["DayOfWeek"] >= 6
).astype(int)

In [25]:
air_xgb_model= XGBClassifier(random_state=1, enable_categorical=True)

In [26]:
air_xgb_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [27]:
print(air_xgb_model.predict(X.head(5)))

[0 0 0 0 0]


In [28]:
print(y.head(5))

0    0
1    0
2    0
3    0
4    0
dtype: int64


In [29]:
type_prediction=air_xgb_model.predict(X.head(5))

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

air_xgb_model.fit(X_train, y_train)

type_prediction = air_xgb_model.predict(X_test)

In [31]:
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
val_prediction = air_xgb_model.predict(X_test)

In [18]:
print(mean_absolute_percentage_error(y_test, val_prediction))

79883858090096.16


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:")
print(accuracy_score(y_test, type_prediction))

print("\nClassification Report:")
print(classification_report(y_test, type_prediction))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, type_prediction))

Accuracy:
0.9397486485027712

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.98      0.96   2188241
           1       0.90      0.80      0.85    588057
           2       0.99      0.88      0.94     39838

    accuracy                           0.94   2816136
   macro avg       0.95      0.89      0.92   2816136
weighted avg       0.94      0.94      0.94   2816136


Confusion Matrix:
[[2138364   49802      75]
 [ 115067  472878     112]
 [   2614    2006   35218]]


In [23]:
from pathlib import Path

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Only keep flights that have a valid DepDel15 target
model_df = df.dropna(subset=["DepDel15"]).copy()

# Features
X_model = model_df[feature_cols].copy()

# Target
y = model_df["DepDel15"]

# Make categorical columns categorical again
for col in categorical_cols:
    X_model[col] = X_model[col].astype("category")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create model
depdel15_model = XGBClassifier(
    random_state=1,
    enable_categorical=True
)

# Train
depdel15_model.fit(X_train, y_train)

# Predict
depdel15_prediction = depdel15_model.predict(X_test)

# Evaluate
print("DepDel15 Accuracy:", accuracy_score(y_test, depdel15_prediction))
print(classification_report(y_test, depdel15_prediction))

# Save
depdel15_model.save_model(
    MODEL_DIR / "depdel15_model.json"
)

NameError: name 'df' is not defined

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Only keep flights that have a valid DepDel15 target
model_df = df.dropna(subset=["DepDel15"]).copy()

# Features
X_model = model_df[feature_cols].copy()

# Target
y = model_df["DepDel15"]

# Make categorical columns categorical again
for col in categorical_cols:
    X_model[col] = X_model[col].astype("category")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create model
depdel15_model = XGBClassifier(
    random_state=1,
    enable_categorical=True
)

# Train
depdel15_model.fit(X_train, y_train)

# Predict
depdel15_prediction = depdel15_model.predict(X_test)

# Evaluate
print("DepDel15 Accuracy:", accuracy_score(y_test, depdel15_prediction))
print(classification_report(y_test, depdel15_prediction))

# Save
depdel15_model.save_model(
    MODEL_DIR / "depdel15_model.json"
)

DepDel15 Accuracy: 0.94040083173016
              precision    recall  f1-score   support

         0.0       0.95      0.98      0.96   2188831
         1.0       0.90      0.81      0.85    588993

    accuracy                           0.94   2777824
   macro avg       0.93      0.89      0.91   2777824
weighted avg       0.94      0.94      0.94   2777824



In [27]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Only keep flights that have a valid Cancelled target
model_df = df.dropna(subset=["Cancelled"]).copy()

# Features
X_model = model_df[feature_cols].copy()

# Target
y = model_df["Cancelled"]

# Make categorical columns categorical again
for col in categorical_cols:
    X_model[col] = X_model[col].astype("category")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create model
cancelled_model = XGBClassifier(
    random_state=1,
    enable_categorical=True
)

# Train
cancelled_model.fit(X_train, y_train)

# Predict
cancelled_prediction = cancelled_model.predict(X_test)

# Evaluate
print("Cancelled Accuracy:", accuracy_score(y_test, cancelled_prediction))
print(classification_report(y_test, cancelled_prediction))

# Save
cancelled_model.save_model(
    MODEL_DIR / "cancelled_model.json"
)

Cancelled Accuracy: 0.9997042046264811
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00   2776298
         1.0       1.00      0.98      0.99     39838

    accuracy                           1.00   2816136
   macro avg       1.00      0.99      0.99   2816136
weighted avg       1.00      1.00      1.00   2816136



In [29]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Only keep flights that have a valid Diverted target
model_df = df.dropna(subset=["Diverted"]).copy()

# Features
X_model = model_df[feature_cols].copy()

# Target
y = model_df["Diverted"]

# Make categorical columns categorical again
for col in categorical_cols:
    X_model[col] = X_model[col].astype("category")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create model
diverted_model = XGBClassifier(
    random_state=1,
    enable_categorical=True
)

# Train
diverted_model.fit(X_train, y_train)

# Predict
diverted_prediction = diverted_model.predict(X_test)

# Evaluate
print("Diverted Accuracy:", accuracy_score(y_test, diverted_prediction))
print(classification_report(y_test, diverted_prediction))

# Save
diverted_model.save_model(
    MODEL_DIR / "diverted_model.json"
)

Diverted Accuracy: 0.9976414491345589
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00   2808785
         1.0       0.67      0.19      0.29      7351

    accuracy                           1.00   2816136
   macro avg       0.84      0.59      0.65   2816136
weighted avg       1.00      1.00      1.00   2816136



In [32]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# Only keep flights that have a valid DepDelayMinutes target
model_df = df.dropna(subset=["DepDelayMinutes"]).copy()

# Features
X_model = model_df[feature_cols].copy()

# Target
y = model_df["DepDelayMinutes"]

# Make categorical columns categorical again
for col in categorical_cols:
    X_model[col] = X_model[col].astype("category")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.2,
    random_state=42
)

# Create model
depdelay_minutes_model = XGBRegressor(
    random_state=1,
    enable_categorical=True
)

# Train
depdelay_minutes_model.fit(X_train, y_train)

# Predict
depdelay_minutes_prediction = depdelay_minutes_model.predict(X_test)

# Evaluate
mae = mean_absolute_error(
    y_test,
    depdelay_minutes_prediction
)

print("DepDelayMinutes MAE:", mae)

# Save
depdelay_minutes_model.save_model(
    MODEL_DIR / "depdelay_minutes_model.json"
)


print("MAE:", mae)
print("Mean actual delay:", y_test.mean())
print("Median actual delay:", y_test.median())
print("Max actual delay:", y_test.max())

DepDelayMinutes MAE: 8.551024563264264
MAE: 8.551024563264264
Mean actual delay: 16.511601526950592
Median actual delay: 0.0
Max actual delay: 3777.0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# Only keep flights that have a valid ArrDelayMinutes target
model_df = df.dropna(subset=["ArrDelayMinutes"]).copy()

# Features
X_model = model_df[feature_cols].copy()

# Target
y = model_df["ArrDelayMinutes"]

# Make categorical columns categorical again
for col in categorical_cols:
    X_model[col] = X_model[col].astype("category")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.2,
    random_state=42
)

# Create model
arrdelay_minutes_model = XGBRegressor(
    random_state=1,
    enable_categorical=True
)

# Train
arrdelay_minutes_model.fit(X_train, y_train)

# Predict
arrdelay_minutes_prediction = arrdelay_minutes_model.predict(X_test)

# Evaluate
mae = mean_absolute_error(
    y_test,
    arrdelay_minutes_prediction
)

print("ArrDelayMinutes MAE:", mae)

# Save
arrdelay_minutes_model.save_model(
    MODEL_DIR / "arrdelay_minutes_model.json"
)

print("MAE:", mae)
print("Mean actual delay:", y_test.mean())
print("Median actual delay:", y_test.median())
print("Max actual delay:", y_test.max())

ArrDelayMinutes MAE: 7.097387100865726
